In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, levene
from statannotations.Annotator import Annotator
from scipy.stats import shapiro, levene
from scipy.stats import skew, kurtosis
from statannotations.Annotator import Annotator


# =========================================================
# CONFIG
# =========================================================
CONDITIONS = ['3g-1gNa-1h', '3g-1gNa-4h', '1gNa-end(8x)']
REAL_INDICES = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
MAX_STEPS = 14400
MODELS = ['RW', 'RW_switch', 'KRW', 'KRW_switch']

PKL_PATH = r"E:\ZHY\code\AssociativeModel\data\condition_dict.pkl"
SIM_OUTPUT_DIR = r"E:\ZHY\code\AssociativeModel\Fitting\SingleConc_resultsV12_KRW_CDF_conditional"
OUT_COMBINED_CSV = os.path.join(SIM_OUTPUT_DIR, "combined_real_and_sim_MeanCTX.csv")
OUT_FIG = os.path.join(SIM_OUTPUT_DIR, "combined_MeanCTX_violin.png")

# 如果你的模拟结果在别的目录，比如 hybrid_v8_output，就改这里
# SIM_OUTPUT_DIR = r"E:\ZHY\code\AssociativeModel\Fitting\hybrid_v8_output"

# =========================================================
# COLOR MAP
# =========================================================
def map_color_dict(key_items, color_map='plasma'):
    import matplotlib.cm as cm
    color_vec_norm = np.linspace(0.01, 0.99, len(key_items) + 1)
    cmap = cm.get_cmap(color_map)
    return {key_items[x]: cmap(float(color_vec_norm[x])) for x in range(len(key_items))}

PREDEFINED_ORDER = [
    "3g-1gNa-1h", "3g-1gNa-4h", "1gNa-end(8x)",
    "5gNa-end(8x)", "3g-5gNa-4h", "3g-5gNa-1h"
]
COLOR_MAP = map_color_dict(PREDEFINED_ORDER, color_map='plasma')
palette_for_conditions = {k: COLOR_MAP[k] for k in CONDITIONS if k in COLOR_MAP}
# =========================================================
# 统计检验选择 —— 按两独立样本流程图决策
# =========================================================
def _safe_shapiro(arr):
    """安全的 Shapiro-Wilk 检验，返回 p 值"""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 3:
        return 0.0
    try:
        return shapiro(arr).pvalue
    except Exception:
        return 0.0


def _i_outlier_stats(arr):
    """计算 IQR 法异常值统计：(1.5IQR 异常点占比, 3IQR 极端点占比, 是否存在极端离群点)"""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 4:
        return {'outlier_pct': 0.0, 'extreme_outlier_pct': 0.0, 'has_extreme': False}
    q1 = np.percentile(arr, 25)
    q3 = np.percentile(arr, 75)
    iqr = q3 - q1
    if iqr == 0:
        return {'outlier_pct': 0.0, 'extreme_outlier_pct': 0.0, 'has_extreme': False}
    lower_15 = q1 - 1.5 * iqr
    upper_15 = q3 + 1.5 * iqr
    lower_3 = q1 - 3.0 * iqr
    upper_3 = q3 + 3.0 * iqr
    n = len(arr)
    outlier_count = int(np.sum((arr < lower_15) | (arr > upper_15)))
    extreme_count = int(np.sum((arr < lower_3) | (arr > upper_3)))
    return {
        'outlier_pct': outlier_count / n * 100,
        'extreme_outlier_pct': extreme_count / n * 100,
        'has_extreme': extreme_count > 0
    }


def _select_statistical_test(groups, group_names=None):
    """
    按照流程图选择最优的两独立样本统计检验方法。

    流程图:
      0. 兜底条件(极端离群点>3IQR / 离群点占比>20% / n<10且严重畸形) → Bootstrap (降级MW)
      1. 样本量分流: 大样本 n>=30 vs 小样本 10<=n<30
      2. 正态性: 大样本 α=0.05 / 小样本 α=0.1
      3. 方差齐性: Levene 检验
      4. 分布形态相似度: 偏态方向 + 方差比

    groups: list of np.array, 每组为一个独立样本
    group_names: optional list of str, 组名(用于诊断输出)

    Returns:
        test_method: str — 适用于 statannotations 的检验方法名
        reason: str — 选择原因
    """
    if group_names is None:
        group_names = [f"Group{i + 1}" for i in range(len(groups))]

    # 清洗数据
    valid = []
    valid_names = []
    for i, g in enumerate(groups):
        arr = np.asarray(g, dtype=float)
        arr = arr[np.isfinite(arr)]
        if len(arr) > 0:
            valid.append(arr)
            valid_names.append(group_names[i])

    if len(valid) < 2:
        return 'Mann-Whitney', '仅一组有效数据，默认使用 Mann-Whitney'

    ns = [len(g) for g in valid]
    min_n = min(ns)
    max_n = max(ns)

    # ======== Step 0: 兜底条件 ========

    # 0a. 极端离群点 (>3 IQR)
    for i, g in enumerate(valid):
        ostats = _i_outlier_stats(g)
        if ostats['has_extreme']:
            return 'Mann-Whitney', \
                f"兜底: {valid_names[i]}存在极端离群点(>3IQR) → 建议Bootstrap, 降级使用Mann-Whitney"

    # 0b. 离群点(>1.5 IQR)占比 >20%
    for i, g in enumerate(valid):
        ostats = _i_outlier_stats(g)
        if ostats['outlier_pct'] > 20:
            return 'Mann-Whitney', \
                f"兜底: {valid_names[i]}离群点占比{ostats['outlier_pct']:.0f}%>20% → 建议Bootstrap, 降级使用Mann-Whitney"

    # 0c. 每组 n<10 且分布严重畸形(|偏度|>2.5 或 |峰度|>7)
    if max_n < 10:
        all_malformed = True
        for g in valid:
            if len(g) >= 3:
                try:
                    sk = skew(g)
                    kt = kurtosis(g)
                    if not (abs(sk) > 2.5 or abs(kt) > 7.0):
                        all_malformed = False
                except Exception:
                    pass
            else:
                all_malformed = False
        if all_malformed:
            return 'Mann-Whitney', \
                '兜底: 每组n<10且分布严重畸形 → 建议Bootstrap, 降级使用Mann-Whitney'

    # ======== Step 1: 样本量评估 ========
    all_large = all(n >= 30 for n in ns)
    any_very_small = any(n < 10 for n in ns)

    # ======== Step 2: 正态性评估 ========
    normal_strict = True   # α=0.05
    normal_relaxed = True  # α=0.1
    obviously_non_normal = False
    skewness_vals = []
    kurtosis_vals = []
    variance_vals = []

    for g in valid:
        if len(g) >= 3:
            try:
                sp = _safe_shapiro(g)
                sk = skew(g)
                kt = kurtosis(g)
                skewness_vals.append(sk)
                kurtosis_vals.append(kt)
                variance_vals.append(np.var(g, ddof=1))

                if sp <= 0.05:
                    normal_strict = False
                if sp <= 0.1:
                    normal_relaxed = False
                if abs(sk) > 1.0 or abs(kt) > 3.0:
                    obviously_non_normal = True
            except Exception:
                normal_strict = False
                normal_relaxed = False
        else:
            normal_strict = False
            normal_relaxed = False

    # ======== Step 3: 方差齐性 (Levene) ========
    equal_var = True
    levene_p = 1.0
    if len(valid) >= 2:
        try:
            levene_stat, levene_p = levene(*valid)
            equal_var = levene_p > 0.05
        except Exception:
            equal_var = True

    # ======== Step 4: 分布形态相似度 ========
    shapes_similar = True
    if len(skewness_vals) >= 2:
        has_pos = any(s > 0.2 for s in skewness_vals)
        has_neg = any(s < -0.2 for s in skewness_vals)
        if has_pos and has_neg:
            shapes_similar = False

    if len(variance_vals) >= 2:
        max_var = max(variance_vals)
        min_var = min(variance_vals)
        if min_var > 0 and max_var / max(min_var, 1e-10) > 4:
            shapes_similar = False

    # ======== Step 5: 决策 ========

    # --- 大样本分支 n>=30 ---
    if all_large:
        if normal_strict or not obviously_non_normal:
            if equal_var:
                return 't-test_ind', \
                    f'大样本(n≥30), 近似正态, 方差齐(Levene p={levene_p:.3f}) → 独立样本t检验'
            else:
                return 't-test_welch', \
                    f'大样本(n≥30), 近似正态, 方差不齐(Levene p={levene_p:.3f}) → Welch t检验'
        else:
            if shapes_similar:
                return 'Mann-Whitney', \
                    '大样本(n≥30), 明显非正态, 两组分布形态相似 → Mann-Whitney U检验'
            else:
                return 'Brunner-Munzel', \
                    '大样本(n≥30), 明显非正态, 两组分布形态差异大 → Brunner-Munzel检验'

    # --- 小样本分支 10<=n<30 ---
    else:
        if normal_relaxed:
            if min_n < 5:
                return 'Mann-Whitney', \
                    f'小样本(n最小={min_n}<5), 虽正态但样本过小 → 保守使用Mann-Whitney'
            if equal_var:
                return 't-test_ind', \
                    f'小样本(10≤n<30), 正态(α=0.1), 方差齐(Levene p={levene_p:.3f}) → 独立样本t检验'
            else:
                return 't-test_welch', \
                    f'小样本(10≤n<30), 正态(α=0.1), 方差不齐(Levene p={levene_p:.3f}) → Welch t检验'
        else:
            if shapes_similar:
                has_obvious_outliers = False
                for g in valid:
                    ostats = _i_outlier_stats(g)
                    if ostats['outlier_pct'] > 15:
                        has_obvious_outliers = True
                        break
                if has_obvious_outliers or any_very_small:
                    return 'Mann-Whitney', \
                        '小样本(10≤n<30), 非正态, 形态相似但异常值明显/样本过小 → 建议Bootstrap, 降级使用Mann-Whitney'
                else:
                    return 'Mann-Whitney', \
                        '小样本(10≤n<30), 非正态, 形态相似且异常值少 → Mann-Whitney U检验'
            else:
                return 'Brunner-Munzel', \
                    '小样本(10≤n<30), 非正态且两组分布形态差异大 → Brunner-Munzel检验'
# =========================================================
# DATA LOADING
# =========================================================
def load_real_per_worm(pkl_path, conditions, df_selection, max_steps=MAX_STEPS):
    with open(pkl_path, "rb") as f:
        condition_dict = pickle.load(f)

    real_results = []
    for cond in conditions:
        if cond not in condition_dict:
            continue

        all_reps = condition_dict[cond]
        for rep_idx in df_selection:
            if rep_idx >= len(all_reps):
                continue

            df_real = all_reps[rep_idx].copy()

            if 'Timestamp' in df_real.columns:
                df_real = df_real[df_real['Timestamp'] <= max_steps]

            if 'CTX' not in df_real.columns:
                continue

            grp = df_real.groupby("ID").agg({
                'CTX': 'mean',
                'speed': 'mean'
            }).reset_index()

            for _, r in grp.iterrows():
                real_results.append({
                    'Model': 'Real Data',
                    'Condition': cond,
                    'ID': f"real_rep{rep_idx}_w{int(r['ID'])}",
                    'Mean_CTX': r['CTX'],
                    'Mean_Speed': r['speed']
                })

    return pd.DataFrame(real_results)


def load_sim_per_worm(output_dir, conditions, models, max_steps=MAX_STEPS):
    sim_results = []
    for cond in conditions:
        cond_outdir = os.path.join(output_dir, cond)

        for model_name in models:
            csv_path = os.path.join(cond_outdir, f"{cond}_{model_name}_trajectories.csv")
            if not os.path.exists(csv_path):
                continue

            df_sim = pd.read_csv(csv_path)

            if 'timestep' in df_sim.columns:
                df_sim = df_sim[df_sim['timestep'] <= max_steps]

            agg = df_sim.groupby('ID').agg({
                'CTX': 'mean',
                'speed': 'mean'
            }).reset_index()

            for _, r in agg.iterrows():
                sim_results.append({
                    'Model': model_name,
                    'Condition': cond,
                    'ID': f"{model_name}_w{int(r['ID'])}",
                    'Mean_CTX': r['CTX'],
                    'Mean_Speed': r['speed']
                })

    return pd.DataFrame(sim_results)

# =========================================================
# PLOTTING
# =========================================================
def plot_models_compare(df_combined, conditions, models_to_plot, out_fig_path=None):
    n = len(models_to_plot)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 6), sharey=True)

    if n == 1:
        axes = [axes]

    cond_order = conditions

    for i, m in enumerate(models_to_plot):
        ax = axes[i]
        sub = df_combined[df_combined['Model'] == m].copy()

        if sub.empty:
            ax.set_title(m)
            ax.axis('off')
            continue

        # violin
        sns.violinplot(
            x='Condition',
            y='Mean_CTX',
            data=sub,
            order=cond_order,
            palette=palette_for_conditions,
            ax=ax,
            inner='box',
            cut=0,
            linewidth=0.9
        )

        # swarm 点
        sns.swarmplot(
            x='Condition',
            y='Mean_CTX',
            data=sub,
            order=cond_order,
            color='black',
            edgecolor='black',
            linewidth=0.1,
            size=3,
            alpha=0.7,
            ax=ax
        )

        # 均值小方块
        means = sub.groupby('Condition')['Mean_CTX'].mean().reindex(cond_order)
        x_positions = []
        y_means = []
        for xi, cond in enumerate(cond_order):
            if cond in means.index and not np.isnan(means.loc[cond]):
                x_positions.append(xi)
                y_means.append(means.loc[cond])

        if x_positions:
            ax.scatter(
                x_positions, y_means,
                marker='s',
                s=22,
                facecolor='white',
                edgecolor='black',
                linewidth=0.6,
                zorder=12
            )
            ax.scatter(
                x_positions, y_means,
                marker='o',
                s=4.5,
                color='black',
                zorder=13
            )

        ax.axhline(0, color='red', linestyle='--', linewidth=1.0, alpha=0.6)
        ax.set_title(f"Model: {m}", fontsize=12)
        ax.set_xlabel('')
        ax.set_ylabel('Mean CTX' if i == 0 else '')

        if ax.get_legend():
            ax.get_legend().remove()

        # 显著性比较
        pairs = [
            ('3g-1gNa-1h', '3g-1gNa-4h'),
            ('3g-1gNa-1h', '1gNa-end(8x)'),
            ('3g-1gNa-4h', '1gNa-end(8x)')
        ]
        valid_pairs = [
            p for p in pairs
            if p[0] in sub['Condition'].unique() and p[1] in sub['Condition'].unique()
        ]

        if valid_pairs:
            try:
                groups = [sub[sub['Condition'] == c]['Mean_CTX'].dropna().values for c in cond_order]
                pair_test, reason = _select_statistical_test(groups, group_names=cond_order)
                print(f"  [{m}] {pair_test} — {reason}")
            except Exception:
                pair_test = 'Mann-Whitney'

            ann = Annotator(
                ax=ax,
                pairs=valid_pairs,
                data=sub,
                x='Condition',
                y='Mean_CTX',
                order=cond_order
            )
            ann.configure(
                test=pair_test,
                text_format='star',
                loc='inside',
                line_height=0.02,
                line_offset=0.03,
                verbose=0
            )

            try:
                ann.apply_and_annotate()
            except Exception as e:
                print(f"Annot fail for {m}: {e}")

        ax.grid(axis='y', alpha=0.25)

    plt.tight_layout()
    if out_fig_path:
        plt.savefig(out_fig_path, dpi=150, bbox_inches='tight')
    plt.show()

# =========================================================
# MAIN
# =========================================================
def main():
    print("Loading real data...")
    df_real_plot = load_real_per_worm(PKL_PATH, CONDITIONS, REAL_INDICES, max_steps=MAX_STEPS)
    print(f"Real samples: {len(df_real_plot)}")

    print("Loading simulated data...")
    df_sim_plot = load_sim_per_worm(SIM_OUTPUT_DIR, CONDITIONS, MODELS, max_steps=MAX_STEPS)
    print(f"Simulated samples: {len(df_sim_plot)}")

    df_plot = pd.concat([df_real_plot, df_sim_plot], ignore_index=True, sort=False)

    os.makedirs(SIM_OUTPUT_DIR, exist_ok=True)
    df_plot.to_csv(OUT_COMBINED_CSV, index=False)
    print(f"Saved combined table to: {OUT_COMBINED_CSV}")
    print(df_plot.groupby(['Model', 'Condition']).size().unstack(fill_value=0))

    models_to_plot = ['Real Data'] + MODELS
    plot_models_compare(
        df_plot,
        CONDITIONS,
        models_to_plot,
        out_fig_path=OUT_FIG
    )

if __name__ == "__main__":
    main()

In [ ]:
# =========================================================
# 新增：基于现有 pkl + trajectory csv 的分布/轨迹对比
# 说明：
# - Real 数据直接从 PKL_PATH 读取（与上面 load_real_per_worm 一致的数据源）
# - Sim 数据从 SingleConc_resultsV6/*_trajectories.csv 读取
# =========================================================

def _load_condition_dict_once(pkl_path=PKL_PATH):
    with open(pkl_path, "rb") as f:
        return pickle.load(f)


def get_real_feature_distribution(condition_dict, cond, feature_name, real_indices=REAL_INDICES, max_steps=MAX_STEPS):
    """按 frame 收集真实数据某特征分布（agl_velocity / CTX）"""
    if cond not in condition_dict:
        return np.array([])

    all_reps = condition_dict[cond]
    chunks = []
    for rep_idx in real_indices:
        if rep_idx >= len(all_reps):
            continue
        df_real = all_reps[rep_idx].copy()
        if "Timestamp" in df_real.columns:
            df_real = df_real[df_real["Timestamp"] <= max_steps]
        if feature_name in df_real.columns:
            vals = df_real[feature_name].dropna().values
            if len(vals) > 0:
                chunks.append(vals)

    if len(chunks) == 0:
        return np.array([])
    return np.concatenate(chunks)


def get_real_net_displacement(condition_dict, cond, real_indices=REAL_INDICES, max_steps=MAX_STEPS):
    """按 worm 计算真实数据 net displacement"""
    if cond not in condition_dict:
        return np.array([])

    all_reps = condition_dict[cond]
    nd_list = []
    for rep_idx in real_indices:
        if rep_idx >= len(all_reps):
            continue
        df_real = all_reps[rep_idx].copy()
        if "Timestamp" in df_real.columns:
            df_real = df_real[df_real["Timestamp"] <= max_steps]

        x_col = "X" if "X" in df_real.columns else ("x" if "x" in df_real.columns else None)
        y_col = "Y" if "Y" in df_real.columns else ("y" if "y" in df_real.columns else None)
        if x_col is None or y_col is None or "ID" not in df_real.columns:
            continue

        for _, g in df_real.groupby("ID"):
            if len(g) > 1:
                start = g.iloc[0][[x_col, y_col]].values.astype(float)
                end = g.iloc[-1][[x_col, y_col]].values.astype(float)
                nd_list.append(np.linalg.norm(end - start))

    return np.array(nd_list)


def load_sim_feature_distribution(output_dir, cond, model, feature_name, max_steps=MAX_STEPS):
    """按 frame 收集模拟数据特征分布"""
    csv_path = os.path.join(output_dir, cond, f"{cond}_{model}_trajectories.csv")
    if not os.path.exists(csv_path):
        return np.array([])
    df = pd.read_csv(csv_path)
    if "timestep" in df.columns:
        df = df[df["timestep"] <= max_steps]
    if feature_name not in df.columns:
        return np.array([])
    return df[feature_name].dropna().values


def load_sim_net_displacement(output_dir, cond, model, max_steps=MAX_STEPS):
    """按 worm 计算模拟数据 net displacement"""
    csv_path = os.path.join(output_dir, cond, f"{cond}_{model}_trajectories.csv")
    if not os.path.exists(csv_path):
        return np.array([])
    df = pd.read_csv(csv_path)
    if "timestep" in df.columns:
        df = df[df["timestep"] <= max_steps]
    if not all(c in df.columns for c in ["x", "y", "ID"]):
        return np.array([])

    nd_list = []
    for _, g in df.groupby("ID"):
        if len(g) > 1:
            start = g.iloc[0][["x", "y"]].values.astype(float)
            end = g.iloc[-1][["x", "y"]].values.astype(float)
            nd_list.append(np.linalg.norm(end - start))
    return np.array(nd_list)


def _plot_single_violin_5groups(data_dict, cond, title, ylabel, save_path):
    order = ["Real", "RW", "RW_switch", "KRW", "KRW_switch"]
    colors = {
        "Real": "gray",
        "RW": "royalblue",
        "RW_switch": "deepskyblue",
        "KRW": "tomato",
        "KRW_switch": "darkred"
    }

    labels = [k for k in order if k in data_dict and len(data_dict[k]) > 0]
    if len(labels) < 2:
        print(f"[SKIP] {cond} {title}: not enough groups")
        return

    data_to_plot = [data_dict[k] for k in labels]

    fig, ax = plt.subplots(figsize=(10, 6))
    parts = ax.violinplot(data_to_plot, showmeans=True, showmedians=True)

    for i, body in enumerate(parts["bodies"]):
        body.set_facecolor(colors.get(labels[i], "lightblue"))
        body.set_edgecolor("black")
        body.set_alpha(0.7)

    for partname in ("cbars", "cmins", "cmaxes", "cmeans", "cmedians"):
        if partname in parts:
            vp = parts[partname]
            vp.set_edgecolor("black")
            vp.set_linewidth(1)

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_title(f"{title}: {cond}", fontsize=12)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


def plot_feature_violin_all_conditions(output_dir, conditions, feature_name, condition_dict):
    for cond in conditions:
        data_dict = {}
        data_dict["Real"] = get_real_feature_distribution(condition_dict, cond, feature_name)
        for m in MODELS:
            data_dict[m] = load_sim_feature_distribution(output_dir, cond, m, feature_name)
        out_path = os.path.join(output_dir, cond, f"{cond}_{feature_name}_violin.png")
        _plot_single_violin_5groups(
            data_dict=data_dict,
            cond=cond,
            title=f"{feature_name} Distribution",
            ylabel=feature_name,
            save_path=out_path
        )


def plot_net_displacement_violin_all_conditions(output_dir, conditions, condition_dict):
    for cond in conditions:
        data_dict = {}
        data_dict["Real"] = get_real_net_displacement(condition_dict, cond)
        for m in MODELS:
            data_dict[m] = load_sim_net_displacement(output_dir, cond, m)
        out_path = os.path.join(output_dir, cond, f"{cond}_net_displacement_violin.png")
        _plot_single_violin_5groups(
            data_dict=data_dict,
            cond=cond,
            title="Net Displacement Distribution",
            ylabel="Net Displacement (pixels)",
            save_path=out_path
        )


def load_real_trajectory_for_plot(condition_dict, cond, real_indices=REAL_INDICES, max_steps=MAX_STEPS):
    """拼接真实轨迹用于画 5-panel"""
    if cond not in condition_dict:
        return pd.DataFrame(columns=["x", "y", "ID"])

    out = []
    all_reps = condition_dict[cond]
    for rep_idx in real_indices:
        if rep_idx >= len(all_reps):
            continue
        d = all_reps[rep_idx].copy()
        if "Timestamp" in d.columns:
            d = d[d["Timestamp"] <= max_steps]
        x_col = "X" if "X" in d.columns else ("x" if "x" in d.columns else None)
        y_col = "Y" if "Y" in d.columns else ("y" if "y" in d.columns else None)
        if x_col is None or y_col is None or "ID" not in d.columns:
            continue
        dd = d[[x_col, y_col, "ID"]].rename(columns={x_col: "x", y_col: "y"}).dropna()
        dd["ID"] = dd["ID"].astype(str) + f"_rep{rep_idx}"
        out.append(dd)

    if len(out) == 0:
        return pd.DataFrame(columns=["x", "y", "ID"])
    return pd.concat(out, ignore_index=True)


def plot_trajectory_5panel(output_dir, conditions, condition_dict):
    width_real, height_real = 4100, 3500
    names = ["Real", "RW", "RW_switch", "KRW", "KRW_switch"]
    colors = {
        "Real": "gray",
        "RW": "royalblue",
        "RW_switch": "deepskyblue",
        "KRW": "tomato",
        "KRW_switch": "darkred",
    }

    for cond in conditions:
        fig, axes = plt.subplots(1, 5, figsize=(30, 6))

        # Real
        d_real = load_real_trajectory_for_plot(condition_dict, cond)
        for _, g in d_real.groupby("ID"):
            axes[0].plot(g["x"], g["y"], lw=0.5, alpha=0.6, color=colors["Real"])
        axes[0].set_title("Real")

        # Sim models
        for i, m in enumerate(MODELS, start=1):
            csv_path = os.path.join(output_dir, cond, f"{cond}_{m}_trajectories.csv")
            if os.path.exists(csv_path):
                d = pd.read_csv(csv_path)
                if "timestep" in d.columns:
                    d = d[d["timestep"] <= MAX_STEPS]
                if all(c in d.columns for c in ["x", "y", "ID"]):
                    for _, g in d.groupby("ID"):
                        axes[i].plot(g["x"], g["y"], lw=0.5, alpha=0.65, color=colors[m])
            axes[i].set_title(m)

        for ax in axes:
            ax.set_xlim(0, width_real)
            ax.set_ylim(0, height_real)
            ax.set_aspect("equal")

        plt.tight_layout()
        out_path = os.path.join(output_dir, cond, f"{cond}_trajectories_5panel_compare.png")
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out_path}")


# -------------------------
# 执行新增分析
# -------------------------
_condition_dict_cache = _load_condition_dict_once(PKL_PATH)

print("\n" + "=" * 60)
print("Agl 分布对比（Real vs 4 models）")
print("=" * 60)
plot_feature_violin_all_conditions(SIM_OUTPUT_DIR, CONDITIONS, "agl_velocity", _condition_dict_cache)

print("\n" + "=" * 60)
print("CTX 分布对比（Real vs 4 models）")
print("=" * 60)
plot_feature_violin_all_conditions(SIM_OUTPUT_DIR, CONDITIONS, "CTX", _condition_dict_cache)

print("\n" + "=" * 60)
print("Net Displacement 分布对比（Real vs 4 models）")
print("=" * 60)
plot_net_displacement_violin_all_conditions(SIM_OUTPUT_DIR, CONDITIONS, _condition_dict_cache)

print("\n" + "=" * 60)
print("轨迹 5-panel 对比（Real vs 4 models）")
print("=" * 60)
plot_trajectory_5panel(SIM_OUTPUT_DIR, CONDITIONS, _condition_dict_cache)

print("\nDone.")

In [ ]:
# =========================================================
# One-Figure Multi-Model Compare (Real + 4 models, each panel has 3 conditions)
# =========================================================

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, levene, skew, kurtosis
from statannotations.Annotator import Annotator


# ---------- helper ----------
def _pick_xy_cols(df):
    x_col = "X" if "X" in df.columns else ("x" if "x" in df.columns else None)
    y_col = "Y" if "Y" in df.columns else ("y" if "y" in df.columns else None)
    return x_col, y_col


def _safe_shapiro(arr):
    """安全的 Shapiro-Wilk 检验，返回 p 值（复用 Cell1 的同名函数逻辑）"""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 3:
        return 0.0
    try:
        return shapiro(arr).pvalue
    except Exception:
        return 0.0


def _i_outlier_stats(arr):
    """IQR 法异常值统计（复用 Cell1 的同名函数逻辑）"""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 4:
        return {'outlier_pct': 0.0, 'extreme_outlier_pct': 0.0, 'has_extreme': False}
    q1 = np.percentile(arr, 25)
    q3 = np.percentile(arr, 75)
    iqr = q3 - q1
    if iqr == 0:
        return {'outlier_pct': 0.0, 'extreme_outlier_pct': 0.0, 'has_extreme': False}
    lower_15 = q1 - 1.5 * iqr
    upper_15 = q3 + 1.5 * iqr
    lower_3 = q1 - 3.0 * iqr
    upper_3 = q3 + 3.0 * iqr
    n = len(arr)
    outlier_count = int(np.sum((arr < lower_15) | (arr > upper_15)))
    extreme_count = int(np.sum((arr < lower_3) | (arr > upper_3)))
    return {
        'outlier_pct': outlier_count / n * 100,
        'extreme_outlier_pct': extreme_count / n * 100,
        'has_extreme': extreme_count > 0
    }


def _auto_pair_test(groups, group_names=None):
    """
    按流程图选择最优统计检验方法（替换原简化版）。

    流程图:
      0. 兜底条件(极端离群点>3IQR / 离群点占比>20% / n<10且严重畸形) → Bootstrap(降级MW)
      1. 样本量分流: 大样本 n>=30 / 小样本 10<=n<30
      2. 正态性: 大样本 α=0.05 / 小样本 α=0.1
      3. 方差齐性: Levene
      4. 分布形态相似度: 偏态方向 + 方差比
    """
    if group_names is None:
        group_names = [f"Group{i + 1}" for i in range(len(groups))]

    valid = []
    valid_names = []
    for i, g in enumerate(groups):
        arr = np.asarray(g, dtype=float)
        arr = arr[np.isfinite(arr)]
        if len(arr) > 0:
            valid.append(arr)
            valid_names.append(group_names[i])

    if len(valid) < 2:
        return 'Mann-Whitney'

    ns = [len(g) for g in valid]
    min_n = min(ns)
    max_n = max(ns)

    # Step 0: 兜底条件
    for i, g in enumerate(valid):
        ostats = _i_outlier_stats(g)
        if ostats['has_extreme']:
            return 'Mann-Whitney'
    for i, g in enumerate(valid):
        ostats = _i_outlier_stats(g)
        if ostats['outlier_pct'] > 20:
            return 'Mann-Whitney'

    if max_n < 10:
        all_malformed = True
        for g in valid:
            if len(g) >= 3:
                try:
                    sk = skew(g)
                    kt = kurtosis(g)
                    if not (abs(sk) > 2.5 or abs(kt) > 7.0):
                        all_malformed = False
                except Exception:
                    pass
            else:
                all_malformed = False
        if all_malformed:
            return 'Mann-Whitney'

    # Step 1: 样本量
    all_large = all(n >= 30 for n in ns)
    any_very_small = any(n < 10 for n in ns)

    # Step 2: 正态性
    normal_strict = True
    normal_relaxed = True
    obviously_non_normal = False
    skewness_vals = []
    variance_vals = []

    for g in valid:
        if len(g) >= 3:
            try:
                sp = _safe_shapiro(g)
                sk = skew(g)
                kt = kurtosis(g)
                skewness_vals.append(sk)
                variance_vals.append(np.var(g, ddof=1))
                if sp <= 0.05:
                    normal_strict = False
                if sp <= 0.1:
                    normal_relaxed = False
                if abs(sk) > 1.0 or abs(kt) > 3.0:
                    obviously_non_normal = True
            except Exception:
                normal_strict = False
                normal_relaxed = False
        else:
            normal_strict = False
            normal_relaxed = False

    # Step 3: 方差齐性
    equal_var = True
    if len(valid) >= 2:
        try:
            _, levene_p = levene(*valid)
            equal_var = levene_p > 0.05
        except Exception:
            equal_var = True

    # Step 4: 分布形态相似度
    shapes_similar = True
    if len(skewness_vals) >= 2:
        has_pos = any(s > 0.2 for s in skewness_vals)
        has_neg = any(s < -0.2 for s in skewness_vals)
        if has_pos and has_neg:
            shapes_similar = False
    if len(variance_vals) >= 2:
        max_var = max(variance_vals)
        min_var = min(variance_vals)
        if min_var > 0 and max_var / max(min_var, 1e-10) > 4:
            shapes_similar = False

    # Step 5: 决策
    if all_large:
        if normal_strict or not obviously_non_normal:
            return 't-test_welch' if not equal_var else 't-test_ind'
        else:
            return 'Brunner-Munzel' if not shapes_similar else 'Mann-Whitney'
    else:
        if normal_relaxed:
            if min_n < 5:
                return 'Mann-Whitney'
            return 't-test_welch' if not equal_var else 't-test_ind'
        else:
            if shapes_similar:
                has_obvious_outliers = False
                for g in valid:
                    ostats = _i_outlier_stats(g)
                    if ostats['outlier_pct'] > 15:
                        has_obvious_outliers = True
                        break
                if has_obvious_outliers or any_very_small:
                    return 'Mann-Whitney'
                else:
                    return 'Mann-Whitney'
            else:
                return 'Brunner-Munzel'


# ---------- metric builders ----------
def build_metric_table(
        metric_name,
        pkl_path=PKL_PATH,
        output_dir=SIM_OUTPUT_DIR,
        conditions=CONDITIONS,
        models=MODELS,
        real_indices=REAL_INDICES,
        max_steps=MAX_STEPS,
        width_real=4100
):
    """
    返回列: Model, Condition, ID, Value
    metric_name 支持:
      - Mean_CTX
      - Mean_Agl
      - Mean_Speed
      - NetDisp
      - MeanDistMid
      - PathLength
      - Tortuosity
    """
    with open(pkl_path, "rb") as f:
        condition_dict = pickle.load(f)

    rows = []

    # Real data from pkl
    for cond in conditions:
        if cond not in condition_dict:
            continue
        reps = condition_dict[cond]

        for rep_idx in real_indices:
            if rep_idx >= len(reps):
                continue
            d = reps[rep_idx].copy()

            if "Timestamp" in d.columns:
                d = d[d["Timestamp"] <= max_steps]

            if "ID" not in d.columns:
                continue

            x_col, y_col = _pick_xy_cols(d)

            for worm_id, g in d.groupby("ID"):
                val = np.nan

                if metric_name == "Mean_CTX":
                    if "CTX" in g.columns:
                        val = g["CTX"].mean()

                elif metric_name == "Mean_Agl":
                    if "agl_velocity" in g.columns:
                        val = g["agl_velocity"].mean()

                elif metric_name == "Mean_Speed":
                    if "speed" in g.columns:
                        val = g["speed"].mean()

                else:
                    # geometry metrics need x/y
                    if x_col is not None and y_col is not None and len(g) > 1:
                        xy = g[[x_col, y_col]].dropna().values.astype(float)
                        if len(xy) > 1:
                            disp = xy[-1] - xy[0]
                            net_disp = float(np.linalg.norm(disp))
                            step_vec = np.diff(xy, axis=0)
                            step_len = np.linalg.norm(step_vec, axis=1)
                            path_len = float(np.sum(step_len))
                            mean_dist_mid = float(np.mean(np.abs(xy[:, 0] - width_real / 2.0)))
                            tort = path_len / max(net_disp, 1e-9)

                            if metric_name == "NetDisp":
                                val = net_disp
                            elif metric_name == "MeanDistMid":
                                val = mean_dist_mid
                            elif metric_name == "PathLength":
                                val = path_len
                            elif metric_name == "Tortuosity":
                                val = tort

                if np.isfinite(val):
                    rows.append({
                        "Model": "Real Data",
                        "Condition": cond,
                        "ID": f"real_rep{rep_idx}_w{worm_id}",
                        "Value": float(val)
                    })

    # Sim data from trajectories csv
    for cond in conditions:
        cond_dir = os.path.join(output_dir, cond)
        for m in models:
            csv_path = os.path.join(cond_dir, f"{cond}_{m}_trajectories.csv")
            if not os.path.exists(csv_path):
                continue

            d = pd.read_csv(csv_path)
            if "timestep" in d.columns:
                d = d[d["timestep"] <= max_steps]
            if "ID" not in d.columns:
                continue

            x_col, y_col = _pick_xy_cols(d)

            for worm_id, g in d.groupby("ID"):
                val = np.nan

                if metric_name == "Mean_CTX":
                    if "CTX" in g.columns:
                        val = g["CTX"].mean()

                elif metric_name == "Mean_Agl":
                    if "agl_velocity" in g.columns:
                        val = g["agl_velocity"].mean()

                elif metric_name == "Mean_Speed":
                    if "speed" in g.columns:
                        val = g["speed"].mean()

                else:
                    if x_col is not None and y_col is not None and len(g) > 1:
                        xy = g[[x_col, y_col]].dropna().values.astype(float)
                        if len(xy) > 1:
                            disp = xy[-1] - xy[0]
                            net_disp = float(np.linalg.norm(disp))
                            step_vec = np.diff(xy, axis=0)
                            step_len = np.linalg.norm(step_vec, axis=1)
                            path_len = float(np.sum(step_len))
                            mean_dist_mid = float(np.mean(np.abs(xy[:, 0] - width_real / 2.0)))
                            tort = path_len / max(net_disp, 1e-9)

                            if metric_name == "NetDisp":
                                val = net_disp
                            elif metric_name == "MeanDistMid":
                                val = mean_dist_mid
                            elif metric_name == "PathLength":
                                val = path_len
                            elif metric_name == "Tortuosity":
                                val = tort

                if np.isfinite(val):
                    rows.append({
                        "Model": m,
                        "Condition": cond,
                        "ID": f"{m}_w{worm_id}",
                        "Value": float(val)
                    })

    return pd.DataFrame(rows)


# ---------- one-figure plot ----------
def plot_metric_one_figure_5panels(
        df_metric,
        metric_name,
        y_label=None,
        conditions=CONDITIONS,
        models_order=None,
        save_path=None,
        add_stats=True
):
    """
    生成 1x5 子图:
      [Real Data, RW, RW_switch, KRW, KRW_switch]
    每个子图 x轴=3个condition, y轴=该metric
    """
    if models_order is None:
        models_order = ["Real Data", "RW", "RW_switch", "KRW", "KRW_switch"]

    # reuse your existing palette_for_conditions
    cond_palette = palette_for_conditions

    fig, axes = plt.subplots(1, len(models_order), figsize=(5 * len(models_order), 6), sharey=True)
    if len(models_order) == 1:
        axes = [axes]

    for i, model_name in enumerate(models_order):
        ax = axes[i]
        sub = df_metric[df_metric["Model"] == model_name].copy()
        sub = sub[np.isfinite(sub["Value"])]

        if sub.empty:
            ax.set_title(f"Model: {model_name}")
            ax.axis("off")
            continue

        sns.violinplot(
            x="Condition",
            y="Value",
            data=sub,
            order=conditions,
            palette=cond_palette,
            ax=ax,
            inner="box",
            cut=0,
            linewidth=0.9
        )

        sns.swarmplot(
            x="Condition",
            y="Value",
            data=sub,
            order=conditions,
            color="black",
            edgecolor="black",
            linewidth=0.1,
            size=2.8,
            alpha=0.7,
            ax=ax
        )

        # mean markers
        means = sub.groupby("Condition")["Value"].mean().reindex(conditions)
        x_pos, y_mean = [], []
        for xi, cond in enumerate(conditions):
            if cond in means.index and np.isfinite(means.loc[cond]):
                x_pos.append(xi)
                y_mean.append(means.loc[cond])

        if x_pos:
            ax.scatter(x_pos, y_mean, marker='s', s=22, facecolor='white',
                       edgecolor='black', linewidth=0.6, zorder=12)
            ax.scatter(x_pos, y_mean, marker='o', s=4.5, color='black', zorder=13)

        # optional significance
        if add_stats:
            pairs = [
                ('3g-1gNa-1h', '3g-1gNa-4h'),
                ('3g-1gNa-1h', '1gNa-end(8x)'),
                ('3g-1gNa-4h', '1gNa-end(8x)')
            ]
            valid_pairs = [p for p in pairs if p[0] in sub['Condition'].unique() and p[1] in sub['Condition'].unique()]

            if valid_pairs:
                groups = [sub[sub["Condition"] == c]["Value"].dropna().values for c in conditions]
                pair_test = _auto_pair_test(groups)

                ann = Annotator(
                    ax=ax, pairs=valid_pairs, data=sub,
                    x='Condition', y='Value', order=conditions
                )
                ann.configure(
                    test=pair_test,
                    text_format='star',
                    loc='inside',
                    line_height=0.02,
                    line_offset=0.03,
                    verbose=0
                )
                try:
                    ann.apply_and_annotate()
                except Exception as e:
                    print(f"Annot fail for {model_name} ({metric_name}): {e}")

        # zero line for CTX-like metrics
        if metric_name.lower().find("ctx") >= 0:
            ax.axhline(0, color='red', linestyle='--', linewidth=1.0, alpha=0.6)

        ax.set_title(f"Model: {model_name}", fontsize=12)
        ax.set_xlabel("")
        if i == 0:
            ax.set_ylabel(y_label if y_label else metric_name)
        else:
            ax.set_ylabel("")
        ax.grid(axis='y', alpha=0.25)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


# ---------- quick run wrappers ----------
def run_one_metric(
        metric_name,
        y_label,
        out_name,
        pkl_path=PKL_PATH,
        output_dir=SIM_OUTPUT_DIR
):
    df_metric = build_metric_table(
        metric_name=metric_name,
        pkl_path=pkl_path,
        output_dir=output_dir,
        conditions=CONDITIONS,
        models=MODELS,
        real_indices=REAL_INDICES,
        max_steps=MAX_STEPS,
        width_real=4100
    )
    if df_metric.empty:
        print(f"[ERROR] No data for metric: {metric_name}")
        return df_metric

    save_path = os.path.join(output_dir, out_name)
    plot_metric_one_figure_5panels(
        df_metric=df_metric,
        metric_name=metric_name,
        y_label=y_label,
        conditions=CONDITIONS,
        models_order=["Real Data", "RW", "RW_switch", "KRW", "KRW_switch"],
        save_path=save_path,
        add_stats=True
    )
    return df_metric


# -------------------------
# Example: run the same layout for multiple metrics
# -------------------------
df_ctx = run_one_metric("Mean_CTX", "Mean CTX", "onefig_Mean_CTX_5panels.png")
df_agl = run_one_metric("Mean_Agl", "Mean Angular Velocity", "onefig_Mean_Agl_5panels.png")
df_nd = run_one_metric("NetDisp", "Net Displacement (pixels)", "onefig_NetDisp_5panels.png")
df_mid = run_one_metric("MeanDistMid", "Mean Distance to Midline (pixels)", "onefig_MeanDistMid_5panels.png")

In [ ]:
# =========================================================
# Low-salt learning curve analysis for 4 models
# Based on LearningSpeed.ipynb with 9 equal-spaced time intervals
# =========================================================

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import mannwhitneyu, shapiro, levene, skew, kurtosis, ttest_ind
import seaborn as sns

# =========================
# CONFIG
# =========================
LOW_SALT_CONDITIONS = ['3g-1gNa-1h', '3g-1gNa-4h', '1gNa-end(8x)']
MODELS = ['RW', 'RW_switch', 'KRW', 'KRW_switch']

PKL_PATH = r"E:\ZHY\code\AssociativeModel\data\condition_dict.pkl"
SIM_OUTPUT_DIR = r"E:\ZHY\code\AssociativeModel\Fitting\SingleConc_resultsV12_KRW_CDF_conditional"

# 统一参数：严格按 LearningSpeed.ipynb 的 9 个等间距时间窗
N_INTERVALS = 9  # 关键：9 个时间窗
MAX_TIME_MINUTES = 12
FPS = 20
WIDTH_PX = 4100
BOARD_WIDTH_MM = 10

OUTPUT_DIR = os.path.join(SIM_OUTPUT_DIR, "LowSalt_LearningCurve_4Models_v2")
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUT_CURVE_FIG = os.path.join(OUTPUT_DIR, "low_salt_learning_curves_4models.png")
OUT_PARAM_FIG = os.path.join(OUTPUT_DIR, "low_salt_parameter_comparison.png")
OUT_CSV = os.path.join(OUTPUT_DIR, "low_salt_learning_curve_parameters.csv")


# =========================
# COLOR MAP
# =========================
def map_color_dict(key_items, color_map='plasma'):
    import matplotlib.cm as cm
    color_vec_norm = np.linspace(0.01, 0.99, len(key_items) + 1)
    cmap = cm.get_cmap(color_map)
    return {key_items[x]: cmap(float(color_vec_norm[x])) for x in range(len(key_items))}


COND_COLORS = map_color_dict(LOW_SALT_CONDITIONS, color_map='plasma')
MODEL_COLORS = {
    'RW': '#4E79A7',
    'RW_switch': '#A0CBE8',
    'KRW': '#F28E2B',
    'KRW_switch': '#E15759'
}


# =========================
# HELPERS
# =========================
def _signed_midline_error(x_coords, width_px, mm_per_px):
    """计算有符号的离中线偏离（保留正负号）"""
    midline = width_px / 2.0
    return (midline - x_coords) * mm_per_px


def exponential_learning_curve(t, a, b, tau):
    """指数学习曲线: y = a + b * exp(-t/tau)"""
    return a + b * np.exp(-t / tau)


def get_significance_stars(p_value):
    """根据p值返回显著性标记"""
    if np.isnan(p_value):
        return 'ns'
    elif p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < 0.05:
        return '*'
    else:
        return 'ns'

# =========================
# 统计检验选择 + 执行 —— 按两独立样本流程图决策
# =========================
def _i_outlier_stats(arr):
    """IQR 法异常值统计"""
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 4:
        return {'outlier_pct': 0.0, 'extreme_outlier_pct': 0.0, 'has_extreme': False}
    q1 = np.percentile(arr, 25)
    q3 = np.percentile(arr, 75)
    iqr = q3 - q1
    if iqr == 0:
        return {'outlier_pct': 0.0, 'extreme_outlier_pct': 0.0, 'has_extreme': False}
    lower_15 = q1 - 1.5 * iqr
    upper_15 = q3 + 1.5 * iqr
    lower_3 = q1 - 3.0 * iqr
    upper_3 = q3 + 3.0 * iqr
    n = len(arr)
    outlier_count = int(np.sum((arr < lower_15) | (arr > upper_15)))
    extreme_count = int(np.sum((arr < lower_3) | (arr > upper_3)))
    return {
        'outlier_pct': outlier_count / n * 100,
        'extreme_outlier_pct': extreme_count / n * 100,
        'has_extreme': extreme_count > 0
    }


def _select_and_run_test(group1, group2):
    """
    按流程图选择并执行两独立样本统计检验。

    流程图:
      0. 兜底条件(极端离群点>3IQR / 离群点占比>20% / n<10且严重畸形) → Bootstrap(降级MW)
      1. 样本量分流: 大样本 n>=30 / 小样本 10<=n<30
      2. 正态性: 大样本 α=0.05 / 小样本 α=0.1
      3. 方差齐性: Levene
      4. 分布形态相似度: 偏态方向 + 方差比

    Returns: (test_name, p_value, reason)
    """
    g1 = np.asarray(group1, dtype=float)
    g2 = np.asarray(group2, dtype=float)
    g1 = g1[np.isfinite(g1)]
    g2 = g2[np.isfinite(g2)]

    if len(g1) < 2 or len(g2) < 2:
        return 'Mann-Whitney', np.nan, '样本量不足，无法检验'

    n1, n2 = len(g1), len(g2)
    ns = [n1, n2]

    # ======== Step 0: 兜底条件 ========
    for i, (arr, name) in enumerate([(g1, 'Group1'), (g2, 'Group2')]):
        ostats = _i_outlier_stats(arr)
        if ostats['has_extreme']:
            try:
                _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                return 'Mann-Whitney', p, f'兜底: {name}存在极端离群点(>3IQR) → 降级MW'
            except Exception:
                return 'Mann-Whitney', np.nan, '兜底: 极端离群点, 检验失败'
        if ostats['outlier_pct'] > 20:
            try:
                _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                return 'Mann-Whitney', p, f'兜底: {name}离群点占比{ostats["outlier_pct"]:.0f}%>20% → 降级MW'
            except Exception:
                return 'Mann-Whitney', np.nan, '兜底: 离群点过多, 检验失败'

    if max(ns) < 10:
        all_malformed = True
        for arr in [g1, g2]:
            if len(arr) >= 3:
                try:
                    sk = skew(arr)
                    kt = kurtosis(arr)
                    if not (abs(sk) > 2.5 or abs(kt) > 7.0):
                        all_malformed = False
                except Exception:
                    pass
            else:
                all_malformed = False
        if all_malformed:
            try:
                _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                return 'Mann-Whitney', p, '兜底: 每组n<10且分布严重畸形 → 降级MW'
            except Exception:
                return 'Mann-Whitney', np.nan, '兜底检验失败'

    # ======== Step 1: 样本量评估 ========
    all_large = all(n >= 30 for n in ns)
    any_very_small = any(n < 10 for n in ns)

    # ======== Step 2: 正态性评估 ========
    normal_strict = True
    normal_relaxed = True
    obviously_non_normal = False
    skewness_vals = []
    variance_vals = []

    for arr in [g1, g2]:
        if len(arr) >= 3:
            try:
                # Shapiro 最多支持 5000 个样本
                if len(arr) <= 5000:
                    sp = shapiro(arr).pvalue
                else:
                    sp = 1.0  # 大样本跳过Shapiro, 依赖偏度/峰度判断
                sk = skew(arr)
                kt = kurtosis(arr)
                skewness_vals.append(sk)
                variance_vals.append(np.var(arr, ddof=1))
                if sp <= 0.05:
                    normal_strict = False
                if sp <= 0.1:
                    normal_relaxed = False
                if abs(sk) > 1.0 or abs(kt) > 3.0:
                    obviously_non_normal = True
            except Exception:
                normal_strict = False
                normal_relaxed = False
        else:
            normal_strict = False
            normal_relaxed = False

    # ======== Step 3: 方差齐性 (Levene) ========
    equal_var = True
    levene_p = 1.0
    try:
        _, levene_p = levene(g1, g2)
        equal_var = levene_p > 0.05
    except Exception:
        equal_var = True

    # ======== Step 4: 分布形态相似度 ========
    shapes_similar = True
    if len(skewness_vals) >= 2:
        has_pos = any(s > 0.2 for s in skewness_vals)
        has_neg = any(s < -0.2 for s in skewness_vals)
        if has_pos and has_neg:
            shapes_similar = False
    if len(variance_vals) >= 2:
        max_var = max(variance_vals)
        min_var = min(variance_vals)
        if min_var > 0 and max_var / max(min_var, 1e-10) > 4:
            shapes_similar = False

    # ======== Step 5: 决策 + 执行检验 ========

    # --- 大样本分支 n>=30 ---
    if all_large:
        if normal_strict or not obviously_non_normal:
            if equal_var:
                try:
                    stat, p = ttest_ind(g1, g2, equal_var=True)
                    return 't-test', p, f'大样本(n={n1},{n2}), 近似正态, 方差齐(Levene p={levene_p:.3f}) → t检验'
                except Exception:
                    pass
            else:
                try:
                    stat, p = ttest_ind(g1, g2, equal_var=False)
                    return 'Welch t-test', p, f'大样本(n={n1},{n2}), 近似正态, 方差不齐(Levene p={levene_p:.3f}) → Welch t检验'
                except Exception:
                    pass
        else:
            if shapes_similar:
                try:
                    _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                    return 'Mann-Whitney', p, f'大样本(n={n1},{n2}), 明显非正态, 分布形态相似 → MW U检验'
                except Exception:
                    pass
            else:
                try:
                    from scipy.stats import brunnermunzel
                    stat, p = brunnermunzel(g1, g2)
                    return 'Brunner-Munzel', p, f'大样本(n={n1},{n2}), 明显非正态, 分布形态差异大 → BM检验'
                except Exception:
                    try:
                        _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                        return 'Mann-Whitney', p, '大样本, BM不可用 → 降级MW'
                    except Exception:
                        pass

    # --- 小样本分支 10<=n<30 ---
    else:
        if normal_relaxed:
            if min(ns) < 5:
                try:
                    _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                    return 'Mann-Whitney', p, f'小样本(n最小={min(ns)}<5), 虽正态但样本过小 → 保守MW'
                except Exception:
                    pass
            elif equal_var:
                try:
                    stat, p = ttest_ind(g1, g2, equal_var=True)
                    return 't-test', p, f'小样本(n={n1},{n2}), 正态(α=0.1), 方差齐 → t检验'
                except Exception:
                    pass
            else:
                try:
                    stat, p = ttest_ind(g1, g2, equal_var=False)
                    return 'Welch t-test', p, f'小样本(n={n1},{n2}), 正态(α=0.1), 方差不齐 → Welch t检验'
                except Exception:
                    pass
        else:
            if shapes_similar:
                has_obvious_outliers = False
                for arr in [g1, g2]:
                    ostats = _i_outlier_stats(arr)
                    if ostats['outlier_pct'] > 15:
                        has_obvious_outliers = True
                        break
                if has_obvious_outliers or any_very_small:
                    try:
                        _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                        return 'Mann-Whitney', p, '小样本, 非正态, 形态相似但异常值明显/样本过小 → 降级MW'
                    except Exception:
                        pass
                else:
                    try:
                        _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                        return 'Mann-Whitney', p, f'小样本(n={n1},{n2}), 非正态, 形态相似且异常值少 → MW U检验'
                    except Exception:
                        pass
            else:
                try:
                    from scipy.stats import brunnermunzel
                    stat, p = brunnermunzel(g1, g2)
                    return 'Brunner-Munzel', p, f'小样本(n={n1},{n2}), 非正态, 分布形态差异大 → BM检验'
                except Exception:
                    try:
                        _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                        return 'Mann-Whitney', p, '小样本, BM不可用 → 降级MW'
                    except Exception:
                        pass

    # 最终兜底
    try:
        _, p = mannwhitneyu(g1, g2, alternative='two-sided')
        return 'Mann-Whitney', p, '所有路径失败 → 默认MW'
    except Exception:
        return 'Mann-Whitney', np.nan, '所有检验均失败'
# =========================
# DATA EXTRACTION - 严格 9 个等间距时间窗
# =========================
def extract_time_series_from_sim(output_dir, conditions, models,
                                 n_intervals=9,
                                 max_time_minutes=12,
                                 fps=20,
                                 width_px=4100,
                                 board_width_mm=10):
    """
    严格按 LearningSpeed.ipynb：
    - 使用 linspace 生成 n_intervals+1 个等间距边界
    - 循环 n_intervals 次（就是 9 次）
    - 计算有符号的离中线距离（保留正负号）
    """
    results = {m: {} for m in models}
    max_time_steps = max_time_minutes * 60 * fps
    mm_per_px = board_width_mm / width_px

    for model in models:
        print(f"\n========== Extracting for model: {model} ==========")
        for cond in conditions:
            csv_path = os.path.join(output_dir, cond, f"{cond}_{model}_trajectories.csv")
            if not os.path.exists(csv_path):
                print(f"[SKIP] missing file: {csv_path}")
                continue

            df = pd.read_csv(csv_path)

            if 'timestep' in df.columns:
                df = df[df['timestep'] <= max_time_steps].copy()
            else:
                print(f"[SKIP] no 'timestep' column")
                continue

            # 获取 x 坐标列
            x_col = 'X' if 'X' in df.columns else ('x' if 'x' in df.columns else None)
            if x_col is None:
                print(f"[SKIP] no x/X column")
                continue

            all_time_points = []
            all_values = []

            # 严格按 LearningSpeed 逻辑：linspace 生成 n_intervals+1 个边界
            max_time = min(df['timestep'].max(), max_time_steps)
            time_intervals = np.linspace(0, max_time, n_intervals + 1)

            for i in range(n_intervals):  # 循环 9 次
                t_start = time_intervals[i]
                t_end = time_intervals[i + 1]

                mask = (df['timestep'] >= t_start) & (df['timestep'] < t_end)
                df_window = df.loc[mask]

                if not df_window.empty:
                    # 计算有符号的离中线距离（保留正负号）
                    avg_value = _signed_midline_error(df_window[x_col], width_px, mm_per_px).mean()
                    time_min = (t_start + t_end) / 2 / (60 * fps)

                    all_time_points.append(time_min)
                    all_values.append(avg_value)

            if len(all_time_points) < 3:
                print(f"[SKIP] not enough points")
                continue

            time_array = np.array(all_time_points)
            value_array = np.array(all_values)
            sort_idx = np.argsort(time_array)

            results[model][cond] = {
                'time': time_array[sort_idx],
                'values': value_array[sort_idx]
            }

            print(f"[OK] {model} / {cond}: {len(time_array)} 个时间点（9 个窗）")

    return results


# =========================
# FIT
# =========================
def fit_learning_curve(time_array, value_array):
    """拟合指数学习曲线"""
    initial_guess = [value_array[-1], value_array[0] - value_array[-1], 10]
    bounds = (
        [value_array.min() - 20, -100, 0.1],
        [value_array.max() + 20, 100, 50]
    )

    popt, pcov = curve_fit(
        exponential_learning_curve,
        time_array,
        value_array,
        p0=initial_guess,
        bounds=bounds,
        maxfev=10000
    )

    predicted = exponential_learning_curve(time_array, *popt)
    ss_res = np.sum((value_array - predicted) ** 2)
    ss_tot = np.sum((value_array - np.mean(value_array)) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0

    asymptote = popt[0]
    amplitude = popt[1]
    tau = popt[2]
    learning_rate = 1 / tau if tau > 0 else np.nan
    initial_value = asymptote + amplitude
    t_half = tau * np.log(2) if tau > 0 else np.nan

    # 标准差
    if pcov is not None and np.all(np.isfinite(pcov)):
        tau_std = np.sqrt(max(0, pcov[2, 2]))
        asymptote_std = np.sqrt(max(0, pcov[0, 0]))
        rate_std = tau_std / (tau ** 2) if tau > 0 else np.nan
        t_half_std = tau_std * np.log(2)
    else:
        tau_std = asymptote_std = rate_std = t_half_std = np.nan

    # 限制误差范围
    if not np.isnan(tau_std) and tau_std > abs(tau) * 3:
        tau_std = abs(tau) * 3
    if not np.isnan(rate_std) and rate_std > abs(learning_rate) * 3:
        rate_std = abs(learning_rate) * 3
    if not np.isnan(asymptote_std) and asymptote_std > abs(asymptote) * 3:
        asymptote_std = abs(asymptote) * 3
    if not np.isnan(t_half_std) and t_half_std > abs(t_half) * 3:
        t_half_std = abs(t_half) * 3

    return {
        'predictions': predicted,
        'r_squared': r_squared,
        'asymptote': asymptote,
        'asymptote_std': asymptote_std,
        'learning_rate': learning_rate,
        'learning_rate_std': rate_std,
        'tau': tau,
        'tau_std': tau_std,
        't_half': t_half,
        't_half_std': t_half_std,
        'initial_value': initial_value
    }


def fit_all_models_low_salt(time_series_data):
    """拟合所有 4 个模型"""
    fitted_results = {}

    for model, cond_dict in time_series_data.items():
        print(f"\n{'=' * 80}")
        print(f"Fitting model: {model}")
        print(f"{'=' * 80}")

        for cond, data in cond_dict.items():
            print(f"\nFitting: {model} / {cond}")
            try:
                fit_res = fit_learning_curve(data['time'], data['values'])
                fitted_results[(model, cond)] = {
                    'model': model,
                    'condition': cond,
                    'time': data['time'],
                    'values': data['values'],
                    **fit_res
                }
                print(f"  ✓ R² = {fit_res['r_squared']:.4f}")
                print(f"  ✓ τ = {fit_res['tau']:.2f} min")
                print(f"  ✓ 学习速率 = {fit_res['learning_rate']:.4f}")
                print(f"  ✓ 渐近线 = {fit_res['asymptote']:.4f}")
            except Exception as e:
                print(f"  ✗ Fit failed: {e}")

    return fitted_results


# =========================
# BOOTSTRAP
# =========================
def bootstrap_parameter_distribution(output_dir, model, condition, n_bootstrap=500,
                                     n_intervals=9, max_time_minutes=12, fps=20,
                                     width_px=4100, board_width_mm=10):
    """Bootstrap 参数分布"""
    csv_path = os.path.join(output_dir, condition, f"{condition}_{model}_trajectories.csv")
    if not os.path.exists(csv_path):
        return np.array([]), np.array([])

    df = pd.read_csv(csv_path)

    max_time_steps = max_time_minutes * 60 * fps
    df = df[df['timestep'] <= max_time_steps].copy()

    x_col = 'X' if 'X' in df.columns else ('x' if 'x' in df.columns else None)
    if x_col is None:
        return np.array([]), np.array([])

    learning_rates = []
    asymptotes = []
    mm_per_px = board_width_mm / width_px

    np.random.seed(42)

    max_time = min(df['timestep'].max(), max_time_steps)
    time_intervals = np.linspace(0, max_time, n_intervals + 1)

    for _ in range(n_bootstrap):
        boot_time_points = []
        boot_values = []

        for i in range(n_intervals):
            t_start = time_intervals[i]
            t_end = time_intervals[i + 1]

            win = df[(df['timestep'] >= t_start) & (df['timestep'] < t_end)]
            if win.empty:
                continue

            sampled = win.sample(n=len(win), replace=True)
            avg_value = _signed_midline_error(sampled[x_col], width_px, mm_per_px).mean()
            time_min = (t_start + t_end) / 2 / (60 * fps)

            boot_time_points.append(time_min)
            boot_values.append(avg_value)

        if len(boot_time_points) < 3:
            continue

        boot_time = np.array(boot_time_points)
        boot_value = np.array(boot_values)
        sort_idx = np.argsort(boot_time)
        boot_time = boot_time[sort_idx]
        boot_value = boot_value[sort_idx]

        try:
            bounds = (
                [boot_value.min() - 30, -150, 0.1],
                [boot_value.max() + 30, 150, 60]
            )

            popt, _ = curve_fit(
                exponential_learning_curve,
                boot_time,
                boot_value,
                p0=[boot_value[-1], boot_value[0] - boot_value[-1], 5],
                bounds=bounds,
                maxfev=5000
            )

            tau = popt[2]
            asymp = popt[0]

            if tau > 0 and tau < 60 and abs(asymp) < 500:
                learning_rates.append(1 / tau)
                asymptotes.append(asymp)

        except Exception:
            continue

    return np.array(learning_rates), np.array(asymptotes)


# =========================
# SIGNIFICANCE TESTS
# =========================
def compare_within_low_salt_significance(output_dir, fitted_results, n_bootstrap=500):
    """低盐组内比较"""
    results = {}

    time_pairs = [('1h', '4h'), ('1h', 'end'), ('4h', 'end')]
    cond_map = {
        '1h': '3g-1gNa-1h',
        '4h': '3g-1gNa-4h',
        'end': '1gNa-end(8x)'
    }

    print("\n" + "=" * 90)
    print("Within-group significance tests for low salt")
    print("=" * 90)

    for model in MODELS:
        print(f"\n--- Model: {model} ---")
        model_res = {}

        for tp1, tp2 in time_pairs:
            cond1 = cond_map[tp1]
            cond2 = cond_map[tp2]

            if (model, cond1) not in fitted_results or (model, cond2) not in fitted_results:
                print(f"[SKIP] missing fitted results: {tp1} vs {tp2}")
                continue

            print(f"\n{tp1} vs {tp2}")

            rates_1, asymps_1 = bootstrap_parameter_distribution(
                output_dir, model, cond1, n_bootstrap=n_bootstrap
            )
            rates_2, asymps_2 = bootstrap_parameter_distribution(
                output_dir, model, cond2, n_bootstrap=n_bootstrap
            )

            res = {
                'cond1': cond1,
                'cond2': cond2
            }
            # 学习速率
            if len(rates_1) > 50 and len(rates_2) > 50:
                test_name, p_rate, reason = _select_and_run_test(rates_1, rates_2)
                res['rate_p'] = p_rate
                res['rate_test'] = test_name
                print(
                    f"  learning_rate ({test_name}): "
                    f"{np.mean(rates_1):.4f}±{np.std(rates_1):.4f} vs "
                    f"{np.mean(rates_2):.4f}±{np.std(rates_2):.4f}, "
                    f"p={p_rate:.4f} ({get_significance_stars(p_rate)})"
                )
                print(f"    → {reason}")
            else:
                res['rate_p'] = np.nan

            # 渐近线
            if len(asymps_1) > 50 and len(asymps_2) > 50:
                test_name, p_asymp, reason = _select_and_run_test(asymps_1, asymps_2)
                res['asymp_p'] = p_asymp
                res['asymp_test'] = test_name
                print(
                    f"  asymptote ({test_name}): "
                    f"{np.mean(asymps_1):.4f}±{np.std(asymps_1):.4f} vs "
                    f"{np.mean(asymps_2):.4f}±{np.std(asymps_2):.4f}, "
                    f"p={p_asymp:.4f} ({get_significance_stars(p_asymp)})"
                )
                print(f"    → {reason}")
            else:
                res['asymp_p'] = np.nan

            model_res[f'{tp1}_vs_{tp2}'] = res

        results[model] = model_res

    return results


# =========================
# PLOTTING
# =========================
def plot_learning_curves_4models_low_salt(fitted_results, save_path=None):
    """绘制 4 个模型 × 3 个条件的学习曲线"""
    fig, axes = plt.subplots(len(MODELS), len(LOW_SALT_CONDITIONS),
                             figsize=(18, 14), sharex=False, sharey=False)

    if len(MODELS) == 1:
        axes = np.array([axes])

    for i, model in enumerate(MODELS):
        for j, cond in enumerate(LOW_SALT_CONDITIONS):
            ax = axes[i, j]
            key = (model, cond)

            if key not in fitted_results:
                ax.axis('off')
                continue

            data = fitted_results[key]
            t = data['time']
            y = data['values']
            pred = data['predictions']

            # 按时间点分组绘制误差带
            unique_times = np.unique(t)
            means = []
            stds = []

            for ut in unique_times:
                mask = np.abs(t - ut) < 0.01
                window_vals = y[mask]
                means.append(np.mean(window_vals))
                stds.append(np.std(window_vals, ddof=1))

            means = np.array(means)
            stds = np.array(stds)

            # 填充误差带
            ax.fill_between(unique_times, means - stds, means + stds,
                            color=COND_COLORS[cond], alpha=0.2)

            # 绘制数据点和拟合曲线
            ax.plot(unique_times, means, 'o-',
                    color=COND_COLORS[cond], markersize=5, linewidth=1.5, label='data')
            ax.plot(t, pred, '--',
                    color=COND_COLORS[cond], linewidth=2.2,
                    label=f"fit (R²={data['r_squared']:.3f})")

            ax.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
            ax.grid(True, alpha=0.3)

            ax.set_title(f"{model} / {cond}", fontsize=10, fontweight='bold')
            ax.set_xlabel("Time (min)")
            ax.set_ylabel("Disp to midline (mm)")

            txt = (
                f"τ = {data['tau']:.2f} min\n"
                f"rate = {data['learning_rate']:.4f}\n"
                f"A = {data['asymptote']:.2f}"
            )
            ax.text(0.03, 0.97, txt, transform=ax.transAxes, va='top', ha='left',
                    fontsize=8, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

            ax.legend(fontsize=7, loc='best')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def plot_parameter_summary_with_stars(fitted_results, sig_results, save_path=None):
    """绘制参数对比图"""
    fig, axes = plt.subplots(2, len(MODELS), figsize=(5 * len(MODELS), 10), sharey='row')
    if len(MODELS) == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    time_order = ['1h', '4h', 'end']
    cond_map = {
        '1h': '3g-1gNa-1h',
        '4h': '3g-1gNa-4h',
        'end': '1gNa-end(8x)'
    }

    for col, model in enumerate(MODELS):
        # 收集拟合值
        rate_vals = []
        asymp_vals = []
        for tp in time_order:
            key = (model, cond_map[tp])
            if key in fitted_results:
                rate_vals.append(fitted_results[key]['learning_rate'])
                asymp_vals.append(fitted_results[key]['asymptote'])  # 保留符号
            else:
                rate_vals.append(np.nan)
                asymp_vals.append(np.nan)

        # learning rate
        axr = axes[0, col]
        axr.bar(time_order, rate_vals,
                color=[COND_COLORS[cond_map[tp]] for tp in time_order], alpha=0.8)
        axr.set_title(f"{model}\nLearning rate", fontsize=11, fontweight='bold')
        axr.set_ylabel("Learning rate (1/min)" if col == 0 else "")
        axr.grid(axis='y', alpha=0.3)

        # 显著性标注
        pair_order = [('1h', '4h'), ('1h', 'end'), ('4h', 'end')]
        y_base = np.nanmax(rate_vals) if np.isfinite(np.nanmax(rate_vals)) else 0.1
        y_offsets = [1.15, 1.25, 1.35]

        for idx, (tp1, tp2) in enumerate(pair_order):
            key = f"{tp1}_vs_{tp2}"
            if key in sig_results.get(model, {}):
                p_val = sig_results[model][key].get('rate_p', np.nan)
                stars = get_significance_stars(p_val)
                x1 = time_order.index(tp1)
                x2 = time_order.index(tp2)
                y = y_base * y_offsets[idx]
                axr.plot([x1, x2], [y, y], 'k-', linewidth=1.3)
                axr.text((x1 + x2) / 2, y, stars, ha='center', va='bottom',
                         fontsize=11, fontweight='bold', color='blue')

        # asymptote （有符号）
        axa = axes[1, col]
        axa.bar(time_order, asymp_vals,
                color=[COND_COLORS[cond_map[tp]] for tp in time_order], alpha=0.8)
        axa.set_title(f"{model}\nAsymptote", fontsize=11, fontweight='bold')
        axa.set_ylabel("Asymptote (mm)" if col == 0 else "")
        axa.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
        axa.grid(axis='y', alpha=0.3)

        y_base = np.nanmax(np.abs(asymp_vals)) if np.isfinite(np.nanmax(np.abs(asymp_vals))) else 0.1
        for idx, (tp1, tp2) in enumerate(pair_order):
            key = f"{tp1}_vs_{tp2}"
            if key in sig_results.get(model, {}):
                p_val = sig_results[model][key].get('asymp_p', np.nan)
                stars = get_significance_stars(p_val)
                x1 = time_order.index(tp1)
                x2 = time_order.index(tp2)
                y = y_base * y_offsets[idx]
                axa.plot([x1, x2], [y, y], 'k-', linewidth=1.3)
                axa.text((x1 + x2) / 2, y, stars, ha='center', va='bottom',
                         fontsize=11, fontweight='bold', color='blue')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


# =========================
# CSV EXPORT
# =========================
def export_fitted_results_to_csv(fitted_results, save_path):
    rows = []
    for (model, cond), data in fitted_results.items():
        rows.append({
            'Model': model,
            'Condition': cond,
            'R_Squared': data['r_squared'],
            'Learning_Rate': data['learning_rate'],
            'Tau': data['tau'],
            'T_Half': data['t_half'],
            'Asymptote': data['asymptote'],
            'Initial_Value': data['initial_value']
        })
    df = pd.DataFrame(rows)
    df.to_csv(save_path, index=False)
    print(f"Saved CSV: {save_path}")


# =========================
# MAIN
# =========================
print("=" * 90)
print("Step 1: Extract time-series data (9 equal-spaced intervals)")
print("=" * 90)

time_series_data = extract_time_series_from_sim(
    output_dir=SIM_OUTPUT_DIR,
    conditions=LOW_SALT_CONDITIONS,
    models=MODELS,
    n_intervals=N_INTERVALS,
    max_time_minutes=MAX_TIME_MINUTES,
    fps=FPS,
    width_px=WIDTH_PX,
    board_width_mm=BOARD_WIDTH_MM
)

print("\n" + "=" * 90)
print("Step 2: Fit exponential learning curves")
print("=" * 90)

fitted_results = fit_all_models_low_salt(time_series_data)
print(f"\n✓ Fitted {len(fitted_results)} curves.")

print("\n" + "=" * 90)
print("Step 3: Bootstrap significance tests (within low-salt only)")
print("=" * 90)

sig_results = compare_within_low_salt_significance(SIM_OUTPUT_DIR, fitted_results, n_bootstrap=500)

print("\n" + "=" * 90)
print("Step 4: Plot learning curves")
print("=" * 90)

plot_learning_curves_4models_low_salt(fitted_results, save_path=OUT_CURVE_FIG)

print("\n" + "=" * 90)
print("Step 5: Plot parameter summary with stars")
print("=" * 90)

plot_parameter_summary_with_stars(fitted_results, sig_results, save_path=OUT_PARAM_FIG)

print("\n" + "=" * 90)
print("Step 6: Export CSV")
print("=" * 90)

export_fitted_results_to_csv(fitted_results, OUT_CSV)

print("\nDone.")
print("=" * 90)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

SIM_OUTPUT_DIR = r"E:\ZHY\code\AssociativeModel\Fitting\SingleConc_resultsV12_KRW_CDF_conditional"
# MAX_STEPS = 14400
MAX_STEPS = 2341

TARGET_CONDITION = "1gNa-end(8x)"
TARGET_MODEL = "KRW_switch"   # RW / RW_switch / KRW / KRW_switch
TARGET_ID = None        # None 表示自动取第一个 ID


def _pick_xy_cols(df):
    x_col = "X" if "X" in df.columns else ("x" if "x" in df.columns else None)
    y_col = "Y" if "Y" in df.columns else ("y" if "y" in df.columns else None)
    return x_col, y_col


def load_model_csv(output_dir, condition, model, max_steps=MAX_STEPS):
    csv_path = os.path.join(output_dir, condition, f"{condition}_{model}_trajectories.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Missing file: {csv_path}")

    df = pd.read_csv(csv_path)

    if "timestep" in df.columns:
        df = df[df["timestep"] <= max_steps].copy()

    return df


def get_available_ids(df):
    if "ID" not in df.columns:
        return []
    ids = df["ID"].dropna().unique().tolist()
    try:
        ids = sorted(ids)
    except Exception:
        pass
    return ids


def select_specific_id(df, target_id=None):
    if "ID" not in df.columns:
        raise ValueError("No ID column found.")

    ids = get_available_ids(df)
    if len(ids) == 0:
        raise ValueError("No valid IDs found.")

    if target_id is None:
        target_id = ids[0]

    sub = df[df["ID"] == target_id].copy()
    if sub.empty:
        raise ValueError(f"ID {target_id} not found.")

    if "timestep" in sub.columns:
        sub = sub.sort_values("timestep").reset_index(drop=True)
    elif "Timestamp" in sub.columns:
        sub = sub.sort_values("Timestamp").reset_index(drop=True)

    x_col, y_col = _pick_xy_cols(sub)
    if x_col is None or y_col is None:
        raise ValueError("No X/Y columns found.")

    return sub, target_id, x_col, y_col


def plot_single_trajectory(sub, x_col, y_col, target_condition, target_model, target_id, save_dir=None):
    x = sub[x_col].values
    y = sub[y_col].values

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(x, y, lw=1.2, color="royalblue")
    ax.scatter(x[0], y[0], c="green", s=60, label="start", zorder=3)
    ax.scatter(x[-1], y[-1], c="red", s=60, label="end", zorder=3)

    ax.set_title(f"Trajectory | condition={target_condition} | model={target_model} | ID={target_id}")
    ax.set_xlabel("X (pixels)")
    ax.set_ylabel("Y (pixels)")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"{target_condition}_{target_model}_ID{target_id}_trajectory.png")
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved: {save_path}")

    plt.show()

In [ ]:
df = load_model_csv(SIM_OUTPUT_DIR, TARGET_CONDITION, TARGET_MODEL, max_steps=MAX_STEPS)
print(df.columns)
print(get_available_ids(df))

In [ ]:
sub, target_id, x_col, y_col = select_specific_id(df, TARGET_ID)
print("Selected ID:", target_id)

In [ ]:
plot_single_trajectory(
    sub=sub,
    x_col=x_col,
    y_col=y_col,
    target_condition=TARGET_CONDITION,
    target_model=TARGET_MODEL,
    target_id=target_id,
    save_dir=SIM_OUTPUT_DIR
)